# Allscripts SCM Condition Era Hydration

Build `condition_era` from populated SCM `condition_occurrence` rows using the same 30-day era logic used for Epic and TouchWorks.

## Dependencies
- `_exponent.omop_scm.condition_occurrence` must be populated
- Only mapped conditions (`condition_concept_id <> 0`) with valid person/date rows are eligible


In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.condition_era;

WITH cteConditionTarget AS (
    SELECT
        co.condition_occurrence_id,
        co.person_id,
        co.condition_concept_id,
        co.condition_start_date,
        COALESCE(co.condition_end_date, co.condition_start_date) AS condition_end_date
    FROM _exponent.omop_scm.condition_occurrence co
    WHERE co.condition_concept_id <> 0
      AND co.person_id IS NOT NULL
      AND co.condition_start_date IS NOT NULL
),
cteEndDates AS (
    SELECT
        person_id,
        condition_concept_id,
        date_add(event_date, -30) AS end_date
    FROM (
        SELECT
            person_id,
            condition_concept_id,
            event_date,
            event_type,
            MAX(start_ordinal) OVER (
                PARTITION BY person_id, condition_concept_id
                ORDER BY event_date, event_type
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS start_ordinal,
            ROW_NUMBER() OVER (
                PARTITION BY person_id, condition_concept_id
                ORDER BY event_date, event_type
            ) AS overall_ord
        FROM (
            SELECT
                person_id,
                condition_concept_id,
                condition_start_date AS event_date,
                -1 AS event_type,
                ROW_NUMBER() OVER (
                    PARTITION BY person_id, condition_concept_id
                    ORDER BY condition_start_date
                ) AS start_ordinal
            FROM cteConditionTarget

            UNION ALL

            SELECT
                person_id,
                condition_concept_id,
                date_add(condition_end_date, 30) AS event_date,
                1 AS event_type,
                NULL AS start_ordinal
            FROM cteConditionTarget
        ) rawdata
    ) e
    WHERE (2 * e.start_ordinal) - e.overall_ord = 0
),
cteConditionEnds AS (
    SELECT
        c.person_id,
        c.condition_concept_id,
        c.condition_start_date,
        MIN(e.end_date) AS era_end_date,
        c.condition_occurrence_id
    FROM cteConditionTarget c
    JOIN cteEndDates e
      ON c.person_id = e.person_id
     AND c.condition_concept_id = e.condition_concept_id
     AND e.end_date >= c.condition_start_date
    GROUP BY
        c.person_id,
        c.condition_concept_id,
        c.condition_start_date,
        c.condition_occurrence_id
)
INSERT INTO _exponent.omop_scm.condition_era (
    person_id,
    condition_concept_id,
    condition_era_start_date,
    condition_era_end_date,
    condition_occurrence_count
)
SELECT
    person_id,
    condition_concept_id,
    MIN(condition_start_date) AS condition_era_start_date,
    era_end_date AS condition_era_end_date,
    COUNT(*) AS condition_occurrence_count
FROM cteConditionEnds
GROUP BY person_id, condition_concept_id, era_end_date;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM _exponent.omop_scm.condition_occurrence WHERE condition_concept_id <> 0) AS eligible_condition_occurrence_rows,
  (SELECT COUNT(DISTINCT person_id) FROM _exponent.omop_scm.condition_occurrence WHERE condition_concept_id <> 0) AS eligible_condition_persons,
  (SELECT COUNT(*) FROM _exponent.omop_scm.condition_era) AS condition_era_rows,
  (SELECT COUNT(DISTINCT person_id) FROM _exponent.omop_scm.condition_era) AS condition_era_persons,
  (SELECT COUNT(*) FROM _exponent.omop_scm.condition_era WHERE condition_era_end_date < condition_era_start_date) AS end_before_start_rows;